# 03 — Indexing & Selection

Mastering `loc`, `iloc`, `at`, `iat`, fancy indexing, and MultiIndex is essential
for every data engineering and interview scenario. Wrong indexing is the #1 source
of silent bugs in Pandas code.

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 200

employees = pd.DataFrame({
    'emp_id':           range(1001, 1001 + n),
    'name':             [f'Employee_{i:03d}' for i in range(n)],
    'department':       np.random.choice(['Engineering', 'Sales', 'HR', 'Marketing', 'Finance'], n),
    'salary':           np.random.randint(40000, 150000, n),
    'experience_years': np.random.randint(1, 20, n),
    'join_date':        pd.date_range('2010-01-01', periods=n, freq='W'),
    'rating':           np.random.choice([1.0, 2.0, 3.0, 4.0, 5.0], n),
    'city':             np.random.choice(['Mumbai', 'Delhi', 'Bangalore', 'Pune', 'Chennai'], n),
    'active':           np.random.choice([True, False], n, p=[0.85, 0.15])
})

employees.head(3)

## 1. loc vs iloc — The Core Distinction

- `loc` → **label-based** — uses actual index values
- `iloc` → **position-based** — uses integer positions (0-based)

They differ **significantly** when the index is not a default RangeIndex.

In [ ]:
# Create a DataFrame with non-default index to show the difference clearly
emp_idx = employees.set_index('emp_id')  # index is now 1001, 1002, ...

# loc: access by emp_id LABEL
print("loc[1001]:")
print(emp_idx.loc[1001, ['name', 'department', 'salary']])

print("\niloc[0]:")
print(emp_idx.iloc[0][['name', 'department', 'salary']])  # same row, position 0

In [ ]:
# DANGER: integer index where label != position
s = pd.Series([10, 20, 30], index=[5, 10, 15])

# s[10] — ambiguous! Pandas tries label first → 20
print(f"s[10]      = {s[10]}   (label-based, gives 20)") 
print(f"s.loc[10]  = {s.loc[10]}   (explicitly label)")  
print(f"s.iloc[1]  = {s.iloc[1]}   (position 1)")

## 2. loc — Full Syntax

In [ ]:
# df.loc[row_selector, col_selector]
# Both can be: label, list of labels, slice, boolean array, callable

# Single row label
print(employees.loc[0, 'salary'])

# List of row labels + list of column labels
print(employees.loc[[0, 5, 10], ['name', 'salary', 'department']])

In [ ]:
# Slice with loc — BOTH ends are INCLUSIVE
print(employees.loc[0:4, 'name':'department'])
# Note: 0:4 gives rows 0,1,2,3,4 (5 rows, not 4)

In [ ]:
# Boolean array with loc
high_salary_mask = employees['salary'] > 120000
top_earners = employees.loc[high_salary_mask, ['name', 'department', 'salary']]
print(top_earners.head())

In [ ]:
# Callable with loc — powerful for pipelines
result = employees.loc[
    lambda df: (df['salary'] > 100000) & (df['department'] == 'Engineering'),
    ['name', 'salary', 'experience_years']
]
print(result.head())

## 3. iloc — Full Syntax

In [ ]:
# Single position
print(employees.iloc[0, 2])  # row 0, col 2 (department)

# List of positions
print(employees.iloc[[0, 5, 10], [1, 2, 3]])

In [ ]:
# Slice with iloc — end is EXCLUSIVE
print(employees.iloc[0:4, 1:4])  
# rows 0,1,2,3  (NOT 4)   — 4 rows
# cols 1,2,3    (NOT 4)

In [ ]:
# Negative indexing with iloc
print("Last row:")
print(employees.iloc[-1][['name', 'department', 'salary']])

print("\nLast 3 rows, last 2 columns:")
print(employees.iloc[-3:, -2:])

## 4. at and iat — Scalar Fast Access

In [ ]:
import time

# at: label-based scalar — faster than loc for single value
print(employees.at[5, 'salary'])

# iat: position-based scalar — fastest
print(employees.iat[5, 3])

# Speed comparison on many accesses
n_access = 1000

start = time.perf_counter()
for i in range(n_access):
    _ = employees.loc[i % 200, 'salary']
t_loc = time.perf_counter() - start

start = time.perf_counter()
for i in range(n_access):
    _ = employees.at[i % 200, 'salary']
t_at = time.perf_counter() - start

start = time.perf_counter()
for i in range(n_access):
    _ = employees.iat[i % 200, 3]
t_iat = time.perf_counter() - start

print(f"loc:  {t_loc*1000:.1f}ms")
print(f"at:   {t_at*1000:.1f}ms")
print(f"iat:  {t_iat*1000:.1f}ms")

## 5. Setting Values with loc and iloc

In [ ]:
temp = employees.copy()

# Set scalar
temp.loc[0, 'salary'] = 999999
print(temp.loc[0, ['name', 'salary']])

# Set multiple rows based on condition
temp.loc[temp['department'] == 'HR', 'salary'] = (
    temp.loc[temp['department'] == 'HR', 'salary'] * 1.10
)
print("\nHR salaries after 10% raise:")
print(temp[temp['department'] == 'HR']['salary'].describe())

## 6. Fancy Indexing — Always Returns a Copy

In [ ]:
# Selecting with a list of labels = fancy indexing = always a COPY
subset = employees[['name', 'salary']]

# This does NOT modify the original
subset['salary'] = 0
print("Original salary[0]:", employees.loc[0, 'salary'])  # unchanged
print("Subset salary[0]:  ", subset.loc[0, 'salary'])     # 0

## 7. Chained Indexing — The SettingWithCopyWarning Trap

In [ ]:
# DANGEROUS — chained indexing: two separate __getitem__ calls
# employees[employees['dept'] == 'HR']['salary'] = 50000  # may or may not work

# CORRECT — single loc operation
temp = employees.copy()
temp.loc[temp['department'] == 'HR', 'salary'] = 50000
print(temp[temp['department'] == 'HR']['salary'].unique())

## 8. MultiIndex — Creating and Selecting

In [ ]:
# Create MultiIndex DataFrame — department x city quarterly sales
midx = pd.MultiIndex.from_product(
    [['Engineering', 'Sales', 'Marketing'],
     ['Mumbai', 'Delhi', 'Bangalore']],
    names=['department', 'city']
)

np.random.seed(42)
sales_data = pd.DataFrame(
    np.random.randint(50000, 500000, size=(9, 4)),
    index=midx,
    columns=['Q1', 'Q2', 'Q3', 'Q4']
)
print(sales_data)

In [ ]:
# loc with tuple — access specific level combination
print(sales_data.loc[('Engineering', 'Mumbai')])

# Access all cities for Engineering
print("\nAll Engineering cities:")
print(sales_data.loc['Engineering'])

In [ ]:
# pd.IndexSlice — cleaner syntax for complex MultiIndex slicing
idx = pd.IndexSlice

# Select Engineering and Sales, all cities, Q1 and Q2
print(sales_data.loc[idx[['Engineering', 'Sales'], :], idx[['Q1', 'Q2']]])

In [ ]:
# xs() — cross-section — cleaner for inner level selection
# Select all departments in Mumbai
print(sales_data.xs('Mumbai', level='city'))

## 9. from_tuples and from_product

In [ ]:
# from_tuples — explicit index pairs
tuples = [('India', 'Q1'), ('India', 'Q2'), ('US', 'Q1'), ('US', 'Q2')]
midx2 = pd.MultiIndex.from_tuples(tuples, names=['country', 'quarter'])
s = pd.Series([100, 150, 200, 180], index=midx2, name='revenue')
print(s)
print()
# Access India only
print(s.loc['India'])

## 10. UnsortedIndexError — Critical Interview Trap

In [ ]:
# MultiIndex slicing requires SORTED index
unsorted_idx = pd.MultiIndex.from_tuples([
    ('B', 2), ('A', 1), ('B', 1), ('A', 2)
])
df_unsorted = pd.DataFrame({'val': [10, 20, 30, 40]}, index=unsorted_idx)

# Attempting a slice on unsorted MultiIndex raises UnsortedIndexError
try:
    result = df_unsorted.loc[('A', 1):('B', 1)]
except Exception as e:
    print(f"Error: {type(e).__name__}: {e}")

# Fix: sort_index() first
df_sorted = df_unsorted.sort_index()
print("\nSorted and sliced:")
print(df_sorted.loc[('A', 1):('B', 1)])

## 11. reindex() — Align to New Index

In [ ]:
dept_avg = employees.groupby('department')['salary'].mean()
print(dept_avg)

# Reindex to include a department that doesn't exist
all_depts = ['Engineering', 'Sales', 'HR', 'Marketing', 'Finance', 'Legal']
reindexed = dept_avg.reindex(all_depts, fill_value=0)  # Legal → 0
print("\nWith fill_value=0:")
print(reindexed)

## 12. filter() — Column/Row Label Filtering

In [ ]:
# filter by regex on column names
print(employees.filter(regex='_years|_id').head(3))

# filter by list of columns (like select)
print(employees.filter(items=['name', 'salary', 'city']).head(3))

# filter rows by index using like (substring match)
# Requires string index
s_str = pd.Series([1, 2, 3, 4], index=['alpha', 'beta', 'gamma', 'alpha_2'])
print(s_str.filter(like='alpha'))

## 13. select_dtypes() for Column Targeting

In [ ]:
# Compute correlation only on numeric columns
numeric_cols = employees.select_dtypes(include='number')
print("Numeric columns for correlation:")
print(numeric_cols.corr().round(3))

## 14. Interview: iloc[-1] vs loc[len-1] Gotcha

In [ ]:
# When you drop rows, loc[len-1] may KeyError
dropped = employees.drop(index=[199])  # remove last row
print(f"Rows: {len(dropped)}")

# This would raise KeyError because index label 199 no longer exists
try:
    dropped.loc[199]
except KeyError as e:
    print(f"KeyError: {e}")

# iloc[-1] always gives the last row, regardless of index state
print("\nLast row with iloc[-1]:")
print(dropped.iloc[-1][['name', 'salary']])

## 15. Quick Reference — Indexing Cheatsheet

| Accessor | Row Selector | Col Selector | Slice End |
|----------|-------------|--------------|----------|
| `loc` | label | label | **inclusive** |
| `iloc` | position | position | **exclusive** |
| `at` | label | label | scalar only |
| `iat` | position | position | scalar only |

**Rules:**
- Use `loc` for label-based access
- Use `iloc` for position-based access
- Use `at`/`iat` for single scalar access in loops (faster)
- Never chain indexing for writes: `df['col'][mask] = val` → use `df.loc[mask, 'col'] = val`
- Always `sort_index()` before slicing a MultiIndex